# Problem 1: Multi-agent TLC Eluent Optimization App

In this homework, we build a **multi-agent application** that mimics how a chemist reasons about TLC optimization.

The application:
1. Accepts a reaction SMILES (e.g., `CC(=O)O.CCO>>CC(=O)OCC`)
2. Identifies reactants and products
3. Uses three cooperating agents to iteratively search for the optimal TLC eluent composition

We use the TLC neural network model trained in the previous homework (Class 3-5) as a **TLC emulator** to simulate experimental Rf values.

In [ ]:
import json
import yaml
import numpy as np
from openai import OpenAI
# --------------------------------------------- #
# 导入你在Class 3-5作业里面训练的模型
from model import MBGDNet, smiles_to_fingerprint
# --------------------------------------------- #

## Setup: Load the TLC Emulator

We load the pre-trained TLC neural network model from `TLC_pred.py`. If no saved model exists on disk, it will be trained automatically and saved for future use.

In [ ]:
import os

MODEL_PATH = 'tlc_model.pkl'

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"未找到 {MODEL_PATH}，请先在终端运行 `python model.py` 训练并保存模型"
    )

nn_model = MBGDNet.load(MODEL_PATH)


def predict_rf(model, smiles, hexane_pct, ea_pct):
    """Predict Rf for a single molecule under hexane/EA binary eluent.

    hexane_pct, ea_pct: 0-100 percentages, sum should equal 100.
    对二元 H/EA 体系，将 DCM/MeOH/Et2O 设为 0。
    """
    input_dim = model.weights[0].shape[0]
    fp_size = input_dim - 5
    fp = smiles_to_fingerprint(smiles, fp_type='morgan', radius=4, fpSize=fp_size)
    h_frac = hexane_pct / 100.0
    ea_frac = ea_pct / 100.0
    feature = np.concatenate([fp, [h_frac, ea_frac, 0.0, 0.0, 0.0]]).astype(np.float32)
    predicted_rf = float(model.predict(feature.reshape(1, -1))[0])
    return predicted_rf

In [ ]:
# Quick sanity check: Rf should increase as ethyl acetate fraction increases
test_smiles = 'COC1=CC(Cl)=CC=C1'
print(f"Rf vs. eluent polarity for {test_smiles}:")
for ea in [5, 10, 20, 30, 50, 80]:
    rf = predict_rf(nn_model, test_smiles, 100 - ea, ea)
    print(f"  Hexane {100-ea}% / EA {ea}% -> Rf = {rf:.4f}")

## Q1: Build Three Agents

We implement three agents using the OpenAI-compatible LLM API:

1. **Explanation Agent** - Interprets TLC results and decides if separation is acceptable
2. **Composition Tuning Agent** - Proposes the next eluent composition
3. **Experimentation Agent** - Assembles simulated TLC results into a structured report

In [ ]:
# 用你自己的API与url
client = OpenAI(
    api_key="",
    base_url=""
)

# 笔者这里用的是qwen3.5-397b-a17b
MODEL = 


def parse_json_response(text):
    """Extract JSON from LLM response, handling markdown code blocks."""
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        end_idx = len(lines) - 1
        for i in range(len(lines) - 1, 0, -1):
            if lines[i].strip() == '```':
                end_idx = i
                break
        text = '\n'.join(lines[1:end_idx])
    return json.loads(text)


def call_agent(system_prompt, user_message, retries=2):
    """Call an LLM agent and return parsed JSON response."""
    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.3,
                response_format={"type": "json_object"}
            )
            return parse_json_response(response.choices[0].message.content)
        except Exception as e:
            if attempt < retries:
                print(f"  [Retry {attempt+1}] {e}")
            else:
                raise

In [ ]:
with open('agent_prompts.yaml', 'r', encoding='utf-8') as f:
    agent_config = yaml.safe_load(f)

EXPLANATION_SYSTEM_PROMPT = agent_config['explanation_agent']['system_prompt']
EXPLANATION_INITIAL_PROMPT = agent_config['explanation_agent']['initial_prompt']
COMPOSITION_SYSTEM_PROMPT = agent_config['composition_agent']['system_prompt']
EXPERIMENTATION_SYSTEM_PROMPT = agent_config['experimentation_agent']['system_prompt']

print("Agent prompts loaded from agent_prompts.yaml")

In [ ]:
# 对上述三个agent进行单独测试
demo_product = 'COC1=CC(Cl)=CC=C1'
demo_reactants = ['COC1=CC=CC=C1']
demo_reaction = f"{demo_reactants[0]}>>{demo_product}"

# --- 1. 解释智能体（首次迭代场景：尚无实验数据） ---
print("=" * 60)
print("[Test 1] Explanation Agent (first iteration)")
print("=" * 60)
init_msg = EXPLANATION_INITIAL_PROMPT.format(
    reaction_smiles=demo_reaction,
    product=demo_product,
    reactants=', '.join(demo_reactants),
)
expl_demo = call_agent(EXPLANATION_SYSTEM_PROMPT, init_msg)
print(json.dumps(expl_demo, indent=2, ensure_ascii=False))

# --- 2. 组成调优智能体（基于解释智能体的输出建议初始洗脱剂） ---
print("\n" + "=" * 60)
print("[Test 2] Composition Tuning Agent")
print("=" * 60)
comp_msg = (
    f"Product SMILES: {demo_product}\n"
    f"Reactant SMILES: {', '.join(demo_reactants)}\n\n"
    f"Explanation Agent analysis:\n{json.dumps(expl_demo, indent=2)}\n\n"
    f"History of all compositions tried so far:\n[]"
)
comp_demo = call_agent(COMPOSITION_SYSTEM_PROMPT, comp_msg)
print(json.dumps(comp_demo, indent=2, ensure_ascii=False))

# --- 3. 实验智能体（用模型给出 Rf，然后让智能体整理为报告） ---
print("\n" + "=" * 60)
print("[Test 3] Experimentation Agent")
print("=" * 60)
demo_h = comp_demo['proposed_eluent']['hexane']
demo_ea = comp_demo['proposed_eluent']['ethyl_acetate']
demo_prod_rf = predict_rf(nn_model, demo_product, demo_h, demo_ea)
demo_react_rfs = {r: predict_rf(nn_model, r, demo_h, demo_ea) for r in demo_reactants}

exp_msg = (
    f"Product SMILES: {demo_product}\n"
    f"Reactant SMILES: {', '.join(demo_reactants)}\n"
    f"Eluent tested: Hexane {demo_h}% / Ethyl Acetate {demo_ea}%\n\n"
    f"Simulated Rf values:\n"
    f"  Product ({demo_product}): {demo_prod_rf:.4f}\n"
)
for r in demo_reactants:
    exp_msg += f"  Reactant ({r}): {demo_react_rfs[r]:.4f}\n"
exp_demo = call_agent(EXPERIMENTATION_SYSTEM_PROMPT, exp_msg)
print(json.dumps(exp_demo, indent=2, ensure_ascii=False))

## Q2: Connect the Agents into an Iterative Workflow

The optimization loop works as follows:

1. **User** sends a task request to separate product using TLC (provides a reaction SMILES)
2. **Explanation Agent** interprets the results (first iteration: receives the user task description to start)
3. **Composition Tuning Agent** proposes a better eluent composition
4. **TLC Emulator** (our trained model) predicts Rf values for all compounds
5. **Experimentation Agent** assembles the raw results into a structured report
6. Go back to step 2 and repeat until the stopping criterion is satisfied or the maximum iteration number is reached

In [ ]:
def parse_reaction_smiles(reaction_smiles):
    """Parse reaction SMILES into reactants list and product."""
    parts = reaction_smiles.split('>>')
    reactants = parts[0].split('.')
    products = parts[1].split('.')
    return reactants, products[0]


def run_tlc_optimization(reaction_smiles, max_iterations=5, verbose=True):
    """Run the multi-agent TLC eluent optimization loop."""
    reactants, product = parse_reaction_smiles(reaction_smiles)

    if verbose:
        print(f"Reaction SMILES: {reaction_smiles}")
        print(f"Product: {product}")
        print(f"Reactants: {reactants}")

    history = []
    explanation_result = None
    experiment_result = None

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n{'=' * 70}")
            print(f"  Iteration {iteration + 1}")
            print(f"{'=' * 70}")

        # --- Step 1: Explanation Agent interprets results ---
        # 首轮没有实验数据，跳过 LLM 调用，直接给确定性 stub。
        # 这样 composition agent 拿到稳定的初始上下文，避免首轮 LLM 输出不完整导致 .get() 拿到 None。
        if iteration == 0:
            explanation_result = {
                "separation_assessment": (
                    f"First iteration: no experimental data yet. "
                    f"Task is to find a Hexane/Ethyl Acetate composition such that "
                    f"product Rf is in [0.20, 0.30] and every reactant differs from "
                    f"the product by >0.10 in Rf."
                ),
                "meets_product_rf_target": False,
                "meets_separation_target": False,
                "overall_status": "not_acceptable",
                "next_step_guidance": (
                    "Propose a moderate starting eluent composition "
                    "(e.g. 80% hexane / 20% ethyl acetate) and assess from there."
                )
            }
        else:
            expl_msg = (
                f"Product SMILES: {product}\n"
                f"Reactant SMILES: {', '.join(reactants)}\n"
                f"Target: product Rf 0.20-0.30, each reactant differs from "
                f"product by >0.10 in Rf.\n\n"
                f"Experiment results:\n{json.dumps(experiment_result, indent=2)}\n\n"
                f"History of all iterations so far:\n{json.dumps(history, indent=2)}"
            )
            explanation_result = call_agent(EXPLANATION_SYSTEM_PROMPT, expl_msg)

        if verbose:
            print(f"\n[Explanation Agent]")
            print(f"  Assessment: {explanation_result.get('separation_assessment', '')}")
            print(f"  Product Rf OK? {explanation_result.get('meets_product_rf_target')}")
            print(f"  Separation OK? {explanation_result.get('meets_separation_target')}")
            print(f"  Status: {explanation_result.get('overall_status')}")
            print(f"  Guidance: {explanation_result.get('next_step_guidance', '')}")

        if iteration > 0 and explanation_result.get('overall_status') == 'acceptable':
            if verbose:
                prev = history[-1]
                print(f"\n>>> ACCEPTABLE separation found! "
                      f"Hexane {prev['hexane']}% / EA {prev['ethyl_acetate']}% <<<")
            break

        # --- Step 2: Composition Tuning Agent proposes eluent ---
        comp_msg = (
            f"Product SMILES: {product}\n"
            f"Reactant SMILES: {', '.join(reactants)}\n\n"
            f"Explanation Agent analysis:\n{json.dumps(explanation_result, indent=2)}\n\n"
            f"History of all compositions tried so far:\n{json.dumps(history, indent=2)}"
        )

        composition_result = call_agent(COMPOSITION_SYSTEM_PROMPT, comp_msg)
        if verbose:
            print(f"\n[Composition Agent]")
            print(f"  Proposed: Hexane {composition_result['proposed_eluent']['hexane']}% "
                  f"/ EA {composition_result['proposed_eluent']['ethyl_acetate']}%")
            print(f"  Direction: {composition_result['change_direction']}")
            print(f"  Reason: {composition_result['reason']}")

        if composition_result.get('converged', False) and iteration > 0:
            if verbose:
                print("\n>>> CONVERGED! The current composition satisfies all criteria. <<<")
            break

        hexane = composition_result['proposed_eluent']['hexane']
        ea = composition_result['proposed_eluent']['ethyl_acetate']

        # --- Step 3: TLC Emulator predicts Rf values ---
        product_rf = predict_rf(nn_model, product, hexane, ea)
        reactant_rfs = {r: predict_rf(nn_model, r, hexane, ea) for r in reactants}
        delta_rfs = {r: abs(product_rf - rf) for r, rf in reactant_rfs.items()}

        if verbose:
            print(f"\n[TLC Emulator]")
            print(f"  Product Rf = {product_rf:.4f}")
            for r in reactants:
                print(f"  Reactant {r}: Rf = {reactant_rfs[r]:.4f}, "
                      f"delta_Rf = {delta_rfs[r]:.4f}")

        # --- Step 4: Experimentation Agent assembles report (no history) ---
        exp_msg = (
            f"Product SMILES: {product}\n"
            f"Reactant SMILES: {', '.join(reactants)}\n"
            f"Eluent tested: Hexane {hexane}% / Ethyl Acetate {ea}%\n\n"
            f"Simulated Rf values:\n"
            f"  Product ({product}): {product_rf:.4f}\n"
        )
        for r in reactants:
            exp_msg += f"  Reactant ({r}): {reactant_rfs[r]:.4f}\n"

        experiment_result = call_agent(EXPERIMENTATION_SYSTEM_PROMPT, exp_msg)
        if verbose:
            print(f"\n[Experimentation Agent]")
            print(f"  Summary: {experiment_result.get('experiment_summary', '')}")

        # Record history
        history.append({
            "iteration": iteration + 1,
            "hexane": hexane,
            "ethyl_acetate": ea,
            "product_rf": round(product_rf, 4),
            "reactant_rfs": {k: round(v, 4) for k, v in reactant_rfs.items()},
            "delta_rfs": {k: round(v, 4) for k, v in delta_rfs.items()},
            "status": explanation_result.get('overall_status', 'unknown')
        })

    return history

## Q3: Optimize the TLC Eluent

We test on two widely used reactions in drug synthesis. The goal is to find a hexane/ethyl acetate composition such that:
- Product Rf is between **0.20 and 0.30**
- All reactants have $$|\Delta R_f| > 0.10$$ compared to the product

In [ ]:
# Buchwald-Hartwig 交叉偶联反应
# 反应粗品中仅含反应物1和产物2（按作业说明 1>>2）
reaction_smiles_bh = "[H]c1c([H])c(C([H])([H])[H])c([H])c([H])c1Cl>>[H]c1c([H])c(C([H])([H])[H])c([H])c([H])c1N1C([H])([H])C([H])([H])OC([H])([H])C1([H])[H]"

print("\n" + "#" * 70)
print("# Buchwald-Hartwig 交叉偶联：寻找最佳 Hexane/EA 洗脱剂")
print("#" * 70)
history_bh = run_tlc_optimization(reaction_smiles_bh, max_iterations=8, verbose=True)

print("\n" + "=" * 70)
print("最终历史记录（每次迭代的洗脱剂组成与 Rf）")
print("=" * 70)
print(json.dumps(history_bh, indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt

# matplotlib 中文字体设置，避免中文标题/坐标显示为方框
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ---------------------------------------------------------------- #
# 手动输入数据：把每次迭代的 EA% 与对应的 ΔRf 填进下面两个数组
# 顺序要一一对应
# ---------------------------------------------------------------- #
ea_ratios = [20, 15]                   # 横坐标：乙酸乙酯比例 (%)
delta_rfs = [0.4946, 0.4962]           # 纵坐标：产物与反应物的 |ΔRf|


def plot_delta_rf(ratios, deltas, ratio_label='Ethyl Acetate %',
                  title='ΔRf vs 溶剂比例', threshold=0.10):
    """绘制 ΔRf 随溶剂比例变化的散点折线图。

    ratios: 横坐标数组（溶剂比例，例如 EA%）
    deltas: 纵坐标数组（产物与反应物之间的 |ΔRf|）
    threshold: 作业要求的最小分离阈值，红色虚线标出
    """
    plt.figure(figsize=(8, 5))
    plt.plot(ratios, deltas, 'o-', markersize=10, linewidth=2,
             color='steelblue', label='ΔRf')
    plt.axhline(y=threshold, color='red', linestyle='--',
                label=f'阈值 ΔRf = {threshold}')
    for x, y in zip(ratios, deltas):
        plt.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=10)
    plt.xlabel(ratio_label, fontsize=12)
    plt.ylabel('ΔRf（产物 vs 反应物）', fontsize=12)
    plt.title(title, fontsize=14)
    plt.grid(alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()


plot_delta_rf(ea_ratios, delta_rfs)